# Sandbox Abstraction - Testing Notebook

Interactive notebook for testing the Sandbox abstraction.

## Setup
```bash
git clone https://github.com/mkmeral/sdk-python.git
cd sdk-python && git checkout feat/sandbox-abstraction
pip install -e '.[dev]'
```

In [ ]:
import asyncio
from strands.sandbox import LocalSandbox, DockerSandbox, ExecutionResult, Sandbox, ShellBasedSandbox
print('Imports OK')

## 1. LocalSandbox Basics

In [ ]:
async def test_local_basics():
    sandbox = LocalSandbox(working_dir='/tmp/sandbox-test')
    
    # Execute a command
    print('=== Execute ===')
    async for chunk in sandbox.execute('echo Hello && ls -la /tmp/sandbox-test'):
        if isinstance(chunk, str):
            print(chunk, end='')
        else:
            print(f'exit_code={chunk.exit_code}')
    
    # File operations
    print('\n=== File Ops ===')
    await sandbox.write_file('hello.py', 'print(42)')
    content = await sandbox.read_file('hello.py')
    print(f'Content: {content}')
    print(f'Files: {await sandbox.list_files(".")}')
    
    # Execute code
    print('\n=== Execute Code ===')
    async for chunk in sandbox.execute_code('import sys; print(sys.version)'):
        if isinstance(chunk, str):
            print(chunk, end='')
        else:
            print(f'exit_code={chunk.exit_code}')

await test_local_basics()

## 2. Agent + Sandbox Integration

In [ ]:
from strands import Agent

agent1 = Agent()
print(f'Default: {type(agent1.sandbox).__name__}, dir={agent1.sandbox.working_dir}')

agent2 = Agent(sandbox=LocalSandbox(working_dir='/tmp/custom'))
print(f'Custom: {type(agent2.sandbox).__name__}, dir={agent2.sandbox.working_dir}')

shared = LocalSandbox(working_dir='/tmp/shared')
a = Agent(sandbox=shared)
b = Agent(sandbox=shared)
print(f'Shared same instance: {a.sandbox is b.sandbox}')

## 3. Concurrent Execution

In [ ]:
import time

async def test_concurrent():
    sandbox = LocalSandbox(working_dir='/tmp/concurrent-test')
    
    async def run(name, delay):
        r = await sandbox._execute_to_result(f'sleep {delay} && echo {name}')
        return f'{name}: {r.stdout.strip()}'
    
    start = time.time()
    results = await asyncio.gather(
        run('task1', 0.5), run('task2', 0.3), run('task3', 0.1),
        run('task4', 0.2), run('task5', 0.4),
    )
    elapsed = time.time() - start
    
    for r in results:
        print(r)
    print(f'\nTotal: {elapsed:.2f}s (should be ~0.5s, not 1.5s)')

await test_concurrent()

## 4. Custom Sandbox (ShellBasedSandbox)

In [ ]:
class LoggingSandbox(ShellBasedSandbox):
    def __init__(self, working_dir='/tmp/logging-sandbox'):
        super().__init__()
        self.working_dir = working_dir
        self.log = []
        self._local = LocalSandbox(working_dir=working_dir)
    
    async def start(self):
        await self._local.start()
        self._started = True
    
    async def stop(self):
        await self._local.stop()
        self._started = False
    
    async def execute(self, command, timeout=None):
        self.log.append(command)
        print(f'LOG: {command[:60]}')
        async for chunk in self._local.execute(command, timeout=timeout):
            yield chunk

async def test_custom():
    s = LoggingSandbox()
    await s.write_file('test.txt', 'hello')
    print(f'Content: {await s.read_file("test.txt")}')
    print(f'Files: {await s.list_files(".")}')
    r = await s._execute_code_to_result('print(2+2)')
    print(f'Code: {r.stdout.strip()}')
    print(f'Commands logged: {len(s.log)}')

await test_custom()

## 5. Context Manager and Lifecycle

In [ ]:
async def test_lifecycle():
    s = LocalSandbox(working_dir='/tmp/lifecycle-test')
    print(f'Before: started={s._started}')
    
    r = await s._execute_to_result('echo auto')
    print(f'After auto-start: started={s._started}, out={r.stdout.strip()}')
    
    await s.stop()
    print(f'After stop: started={s._started}')
    
    async with LocalSandbox(working_dir='/tmp/ctx') as ctx:
        print(f'In ctx: started={ctx._started}')
        r = await ctx._execute_to_result('echo context')
        print(f'Out: {r.stdout.strip()}')
    print(f'After ctx: started={ctx._started}')

await test_lifecycle()

## 6. DockerSandbox (requires Docker)

In [ ]:
import shutil
if shutil.which('docker'):
    async def test_docker():
        async with DockerSandbox(image='python:3.12-slim') as s:
            print(f'Container: {s._container_id}')
            r = await s._execute_to_result('python --version')
            print(f'Python: {r.stdout.strip()}')
            await s.write_file('test.py', 'print(42)')
            print(f'File: {await s.read_file("test.py")}')
            async for c in s.execute_code('import os; print(os.getpid())'):
                if isinstance(c, str): print(c, end='')
                else: print(f'exit={c.exit_code}')
    await test_docker()
else:
    print('Docker not available')